# 02 - Exploratory data analysis (EDA)

Every figure answers a specific question tied to a research question. Numbers quoted in the
interim report come from `outputs/results.json`, written by this notebook.

| Figure | Question | Supports |
|---|---|---|
| fig01 | Do delays and cancellations vary through the year? | RQ1, RQ3 |
| fig02-fig03 | Does the *scheduled* departure hour / weekday matter? | RQ1 |
| fig04 | How different are carriers? | RQ1, RQ3 |
| fig05 | What does arrival delay look like (skew, outliers)? | RQ1, cleaning |
| fig06 | What causes delay minutes? | RQ1 context |
| fig07 | Do airports differ? | RQ1 |
| fig08 | Are there cancellation anomalies? | RQ3 |
| fig09-fig11 | How is load factor distributed and how does it relate to delay? | RQ2 |
| fig12 | Is passenger demand trending / seasonal? | RQ4 |

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from labels import MONTH_NAMES, DOW_NAMES, carrier_name, airport_name
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from plot_style import set_style, save_fig
set_style()
MONTHS = MONTH_NAMES
DOW = DOW_NAMES     # BTS: 1 = Monday
blue_seq = LinearSegmentedColormap.from_list("blue_seq", ["#ffffff"] + BLUE_RAMP)
diverge = LinearSegmentedColormap.from_list("div", ["#256abf", "#f0efec", "#e34948"])

flights = pd.read_parquet(PROCESSED / "otp_clean.parquet")
op = flights[flights["operated"] == 1]
rma = pd.read_csv(PROCESSED / "route_month_analysis.csv")
series = pd.read_csv(PROCESSED / "route_monthly_passengers.csv", index_col="period", parse_dates=True)
eda = {"n_flights": int(len(flights)), "n_operated": int(len(op))}
print(f"{len(flights):,} flights | {len(op):,} operated | {len(rma):,} route-months | series {series.shape}")

14,080,680 flights | 13,844,731 operated | 148,241 route-months | series (55, 150)


## Summary statistics

In [2]:
def describe(s, name):
    d = s.dropna().astype(float)
    q = d.quantile([.25, .5, .75, .99])
    return {"Variable": name, "n": int(d.size), "Mean": d.mean(), "SD": d.std(), "Min": d.min(),
            "Q1": q[.25], "Median": q[.5], "Q3": q[.75], "P99": q[.99], "Max": d.max()}
stats = pd.DataFrame([
    describe(op["arr_delay"], "Arrival delay (min)"),
    describe(flights["distance"], "Distance (mi)"),
    describe(flights["crs_elapsed"], "Scheduled elapsed time (min)"),
    describe(rma["load_factor"], "Route-month load factor (%)"),
    describe(rma["late_rate"], "Route-month late rate (%)"),
    describe(rma["cancel_rate"], "Route-month cancel rate (%)"),
    describe(rma["flights_operated"], "Flights per route-month"),
]).round(2)
stats.to_csv(TAB / "table_summary_stats.csv", index=False)
eda["arr_delay_mean"], eda["arr_delay_median"] = float(op["arr_delay"].mean()), float(op["arr_delay"].median())
eda["late_rate_pct"] = float(op["is_late"].mean() * 100)
eda["cancel_rate_pct"] = float(flights["cancelled"].mean() * 100)
eda["diverted_rate_pct"] = float(flights["diverted"].mean() * 100)
stats

,Variable,n,Mean,SD,Min,Q1,Median,Q3,P99,Max
0,Arrival delay (min),13844731,7.80,58.87,-128.00,-15.00,-6.00,10.00,220.00,4336.00
1,Distance (mi),14080680,839.15,599.13,11.00,402.00,689.00,1073.00,2611.00,5095.00
2,Scheduled elapsed time (min),14080677,147.83,72.68,-160.00,95.00,131.00,179.00,375.00,1510.00
3,Route-month load factor (%),148241,79.98,10.33,12.49,74.75,81.90,87.39,95.33,98.76
4,Route-month late rate (%),148241,21.26,11.32,0.00,12.98,19.73,27.85,54.39,96.67
5,Route-month cancel rate (%),148241,1.21,2.39,0.00,0.00,0.00,1.67,11.29,48.00
6,Flights per route-month,148241,84.92,63.56,30.00,39.00,62.00,107.00,315.00,678.00


## Figure 1 - Late arrivals and cancellations by month

In [3]:
op = op.assign(period=op["flight_date"].dt.to_period("M").dt.to_timestamp())
fl = flights.assign(period=flights["flight_date"].dt.to_period("M").dt.to_timestamp())
m_late = op.groupby("period")["is_late"].mean().mul(100)
m_canc = fl.groupby("period")["cancelled"].mean().mul(100)
fig, ax = plt.subplots(2, 1, figsize=(6.5, 4.4), sharex=True)
ax[0].plot(m_late.index, m_late.values, marker="o", ms=3.5, color=PALETTE[0])
ax[0].set_ylabel("Late arrivals\n(% of operated flights)")
ax[1].bar(m_canc.index, m_canc.values, color=PALETTE[1], width=22)
ax[1].set_ylabel("Cancelled flights\n(% of scheduled)"); ax[1].set_xlabel("Month")
for a in ax: a.margins(y=0.16)
pk = m_late.idxmax(); ax[0].annotate(f"{pk:%B %Y}: {m_late[pk]:.1f}%", (pk, m_late[pk]), textcoords="offset points", xytext=(0, 7), ha="center", color=INK)
pc = m_canc.idxmax(); ax[1].annotate(f"{pc:%B %Y}: {m_canc[pc]:.1f}%", (pc, m_canc[pc]), textcoords="offset points", xytext=(14, -4), ha="left", va="center", color=INK)
fig.tight_layout(); save_fig(fig, "fig01_monthly")
lab = lambda ts: f"{ts:%Y-%m}"
eda["monthly_late"] = {lab(k): round(float(v), 2) for k, v in m_late.items()}; eda["monthly_cancel"] = {lab(k): round(float(v), 2) for k, v in m_canc.items()}
eda["late_peak_period"], eda["late_low_period"] = lab(pk), lab(m_late.idxmin())
eda["cancel_peak_period"] = lab(pc)
yr = {}
for y, g in flights.groupby(flights["flight_date"].dt.year):
    o_ = g[g["operated"] == 1]
    yr[str(int(y))] = {"flights": int(len(g)), "late_rate_pct": round(float(o_["is_late"].mean() * 100), 2),
                       "cancel_rate_pct": round(float(g["cancelled"].mean() * 100), 2), "arr_delay_median": float(o_["arr_delay"].median())}
eda["yearly"] = yr; pd.DataFrame(yr).T

,flights,late_rate_pct,cancel_rate_pct,arr_delay_median
2024,7079061.0,20.82,1.36,-6.0
2025,7001619.0,22.31,1.47,-6.0


## Figures 2-3 - Scheduled departure hour and day of week

In [4]:
h = op.groupby("dep_hour")["is_late"].agg(["mean", "size"])
h = h[h["size"] >= 1000]; h["rate"] = h["mean"] * 100
fig, ax = plt.subplots(figsize=(6.5, 3.0))
cols = [PALETTE[1] if i == h["rate"].idxmax() else PALETTE[0] for i in h.index]
ax.bar(h.index.astype(int), h["rate"], color=cols, width=0.75)
ax.set_xlabel("Scheduled departure hour (local time)"); ax.set_ylabel("Late arrivals (%)")
ax.set_xticks(h.index.astype(int))
mx = int(h["rate"].idxmax()); ax.annotate(f"{mx}:00 -> {h['rate'].max():.1f}%", (mx, h["rate"].max()), textcoords="offset points", xytext=(0, 4), ha="center")
fig.tight_layout(); save_fig(fig, "fig02_hour")
eda["hour_rate"] = h["rate"].round(2).to_dict(); eda["hour_peak"] = mx
eda["hour_earliest_rate"] = float(h["rate"].iloc[0]); eda["hour_earliest"] = int(h.index[0])

d = op.groupby("day_of_week")["is_late"].mean().mul(100)
fig, ax = plt.subplots(figsize=(6.5, 2.8))
ax.bar(range(1, 8), d.values, color=[PALETTE[1] if v == d.max() else PALETTE[0] for v in d.values], width=0.7)
ax.set_xticks(range(1, 8)); ax.set_xticklabels(DOW, rotation=30, ha="right", fontsize=8); ax.set_ylabel("Late arrivals (%)"); ax.set_xlabel("Scheduled day of week")
ax.set_ylim(d.min() - 2, d.max() + 1.5)
for i, v in enumerate(d.values): ax.text(i + 1, v + 0.15, f"{v:.1f}", ha="center", fontsize=8, color=INK_2)
fig.tight_layout(); save_fig(fig, "fig03_dow")
eda["dow_rate"] = {DOW[i - 1]: round(float(v), 2) for i, v in d.items()}
eda["dow_peak"], eda["dow_low"] = DOW[int(d.idxmax()) - 1], DOW[int(d.idxmin()) - 1]

## Figure 4 - Carrier differences

In [5]:
c = op.groupby("carrier", observed=True).agg(late=("is_late", "mean"), n=("is_late", "size"))
c["late"] *= 100
cc = flights.groupby("carrier", observed=True).agg(canc=("cancelled", "mean")); cc["canc"] *= 100
c = c.join(cc); c = c[c["n"] >= 5000].sort_values("late")
fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.6), sharey=True)
ylab = [carrier_name(i) for i in c.index]
ax[0].barh(ylab, c["late"], color=PALETTE[0], height=0.65); ax[0].set_xlabel("Late arrivals (%)")
ax[1].barh(ylab, c["canc"], color=PALETTE[1], height=0.65); ax[1].set_xlabel("Cancelled flights (%)")
for a, col in zip(ax, ("late", "canc")):
    for i, v in enumerate(c[col]): a.text(v, i, f" {v:.1f}", va="center", fontsize=8, color=INK_2)
    a.set_xlim(0, c[col].max() * 1.18)
fig.tight_layout(); save_fig(fig, "fig04_carriers")
eda["carriers"] = {str(k): {"late": round(float(r.late), 2), "cancel": round(float(r.canc), 2), "flights": int(r.n)} for k, r in c.iterrows()}
eda["carrier_late_best"], eda["carrier_late_worst"] = str(c["late"].idxmin()), str(c["late"].idxmax())
eda["carrier_cancel_worst"] = str(c["canc"].idxmax()); eda["n_carriers"] = int(len(c))

## Figure 5 - Distribution of arrival delay

In [6]:
ad = op["arr_delay"].astype(float)
clip = ad.clip(-60, 180)
bins = np.arange(-60, 185, 5)
cnt, edges = np.histogram(clip, bins=bins)
centers = edges[:-1] + 2.5
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.bar(centers[centers < 15], cnt[centers < 15] / 1e3, width=4.5, color=PALETTE[0], label="On time or less than 15 minutes late")
ax.bar(centers[centers >= 15], cnt[centers >= 15] / 1e3, width=4.5, color=PALETTE[1], label="Late (15 or more minutes)")
ax.axvline(15, color=INK_2, lw=1, ls="--")
ax.set_xlabel("Arrival delay (minutes; clipped to -60 / 180 for display)"); ax.set_ylabel("Flights (thousands)")
ax.legend(loc="upper right")
fig.tight_layout(); save_fig(fig, "fig05_delay_dist")
eda["pct_early_or_ontime"] = float((ad < 15).mean() * 100); eda["pct_early"] = float((ad < 0).mean() * 100)
eda["p90"], eda["p99"] = float(ad.quantile(.90)), float(ad.quantile(.99))
eda["skew"] = float(ad.clip(-60, 600).skew())
eda["share_over_60"] = float((ad > 60).mean() * 100); eda["share_over_180"] = float((ad > 180).mean() * 100)

## Figure 6 - Where do delay minutes come from?

In [7]:
cause_cols = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
names = {"carrier_delay": "Air carrier", "weather_delay": "Extreme weather", "nas_delay": "National Aviation System",
         "security_delay": "Security", "late_aircraft_delay": "Late-arriving aircraft"}
late_flights = op[op["is_late"] == 1]
mins = late_flights[cause_cols].fillna(0).sum()
share = (mins / mins.sum() * 100).sort_values()
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.barh([names[i] for i in share.index], share.values, color=PALETTE[0], height=0.6)
for i, v in enumerate(share.values): ax.text(v, i, f" {v:.1f}%", va="center", fontsize=8, color=INK_2)
ax.set_xlabel("Share of total delay minutes on late flights (%)"); ax.set_xlim(0, share.max() * 1.15)
fig.tight_layout(); save_fig(fig, "fig06_causes")
eda["cause_share"] = {names[k]: round(float(v), 2) for k, v in share.items()}
eda["cause_top"] = names[share.idxmax()]
eda["cause_flights_with_any_cause_pct"] = float(late_flights[cause_cols].notna().any(axis=1).mean() * 100)

## Figure 7 - Busiest airports

In [8]:
a = op.groupby("origin", observed=True).agg(late=("is_late", "mean"), n=("is_late", "size"))
a = a.sort_values("n", ascending=False).head(15); a["late"] *= 100; a = a.sort_values("late")
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.barh([airport_name(i) for i in a.index], a["late"], color=PALETTE[0], height=0.65)
for i, v in enumerate(a["late"]): ax.text(v, i, f" {v:.1f}", va="center", fontsize=8, color=INK_2)
ax.set_xlabel("Late arrivals (%) for flights departing from the 15 busiest airports"); ax.set_xlim(0, a["late"].max() * 1.15)
fig.tight_layout(); save_fig(fig, "fig07_airports")
eda["airport_best"], eda["airport_worst"] = str(a["late"].idxmin()), str(a["late"].idxmax())
eda["airport_range"] = [float(a["late"].min()), float(a["late"].max())]
eda["airport_rates"] = {str(k): round(float(v), 2) for k, v in a["late"].items()}

## Figure 8 - Daily cancellations (anomaly check for RQ3)

In [9]:
daily = flights.groupby("flight_date").agg(canc=("cancelled", "sum"), n=("cancelled", "size"))
daily["rate"] = daily["canc"] / daily["n"] * 100
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.plot(daily.index, daily["rate"], color=PALETTE[0], lw=0.8, alpha=0.6, label="Daily")
ax.plot(daily.index, daily["rate"].rolling(7, center=True).mean(), color=PALETTE[1], lw=1.8, label="7-day average")
top3 = daily["rate"].nlargest(3)
for dt, v in top3.items():
    ax.annotate(dt.strftime("%B %d, %Y"), (dt, v), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
ax.set_ylabel("Cancelled flights (% of scheduled)"); ax.set_xlabel("Date"); ax.legend(loc="upper right")
fig.tight_layout(); save_fig(fig, "fig08_daily_cancel")
tops = daily.sort_values("rate", ascending=False).head(5)
detail = []
for dt in tops.index:
    day = flights[flights["flight_date"] == dt]
    by = day.groupby("carrier", observed=True)["cancelled"].agg(["sum", "size"]); by["rate"] = by["sum"] / by["size"] * 100
    by = by[by["size"] >= 100].sort_values("rate", ascending=False)
    detail.append({"date": str(dt.date()), "rate_pct": round(float(daily.loc[dt, "rate"]), 2), "cancelled": int(daily.loc[dt, "canc"]),
                   "worst_carrier": str(by.index[0]) if len(by) else None, "worst_carrier_rate_pct": round(float(by["rate"].iloc[0]), 1) if len(by) else None})
eda["top_cancel_days"] = detail
eda["median_daily_cancel_rate"] = float(daily["rate"].median())
pd.DataFrame(detail)

,date,rate_pct,cancelled,worst_carrier,worst_carrier_rate_pct
0,2024-07-19,16.53,3548,G4,58.8
1,2025-01-10,15.98,2856,OH,36.7
2,2024-01-15,15.91,2962,OH,29.0
3,2024-01-16,12.77,2111,YX,36.9
4,2025-11-09,12.13,2480,YX,30.6


## Figures 9-11 - Load factor and its relationship with delay (RQ2 preview)

In [10]:
lf = rma["load_factor"]
fig, ax = plt.subplots(figsize=(6.5, 2.8))
ax.hist(lf, bins=40, color=PALETTE[0], edgecolor="white", linewidth=0.4)
ax.axvline(lf.median(), color=PALETTE[1], lw=1.6); ax.text(lf.median(), ax.get_ylim()[1] * 0.92, f"  median {lf.median():.1f}%", color=INK)
ax.set_xlabel("Route-month load factor (% of seats filled)"); ax.set_ylabel("Route-months")
fig.tight_layout(); save_fig(fig, "fig09_lf_dist")
eda["lf_median"], eda["lf_q1"], eda["lf_q3"] = float(lf.median()), float(lf.quantile(.25)), float(lf.quantile(.75))
eda["lf_skew"] = float(lf.skew())

fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.2))
hb = ax[0].hexbin(rma["load_factor"], rma["late_rate"], gridsize=35, cmap=blue_seq, mincnt=1, extent=(lf.min(), 100, 0, rma["late_rate"].quantile(.995)))
ax[0].set_xlabel("Load factor (%)"); ax[0].set_ylabel("Late-arrival rate (%)")
cb = fig.colorbar(hb, ax=ax[0], pad=0.02); cb.set_label("Route-months", fontsize=8)
rma["lf_decile"] = pd.qcut(rma["load_factor"], 10, labels=False, duplicates="drop")
dec = rma.groupby("lf_decile").agg(lf=("load_factor", "mean"), late=("late_rate", "mean"), sd=("late_rate", "std"), n=("late_rate", "size"))
dec["se"] = dec["sd"] / np.sqrt(dec["n"])
ax[1].errorbar(dec["lf"], dec["late"], yerr=1.96 * dec["se"], marker="o", color=PALETTE[0], capsize=2.5, lw=1.5)
ax[1].set_xlabel("Mean load factor in decile (%)"); ax[1].set_ylabel("Mean late-arrival rate (%)")
fig.tight_layout(); save_fig(fig, "fig10_lf_vs_late")
eda["decile_table"] = dec.round(3).reset_index().to_dict("records")
eda["decile_first_last"] = [float(dec["late"].iloc[0]), float(dec["late"].iloc[-1])]
dec

,lf,late,sd,n,se
lf_decile,,,,,
0,57.941842,18.586712,10.707930,14825,0.087944
1,70.013800,20.117596,10.491407,14824,0.086169
2,74.700445,20.606366,10.623825,14824,0.087257
3,77.994797,20.951752,10.826067,14824,0.088918
4,80.672526,21.254482,10.956248,14824,0.089987
5,83.042793,21.580792,11.145510,14824,0.091541
6,85.240484,21.905663,11.520214,14824,0.094619
7,87.406511,22.433733,11.746821,14824,0.096480
8,89.731841,22.831612,11.999519,14824,0.098556


In [11]:
cm = rma[["load_factor", "late_rate", "cancel_rate", "flights_operated", "distance_otp", "dep_performed"]].copy()
cm["flights_operated"] = np.log1p(cm["flights_operated"])
cm.columns = ["Load factor", "Late rate", "Cancellation rate", "Log flights", "Distance", "Departures"]
corr = cm.corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.2, 4.2))
im = ax.imshow(corr, cmap=diverge, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=40, ha="right"); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)): ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8, color=INK)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("Spearman correlation", fontsize=8)
fig.tight_layout(); save_fig(fig, "fig11_corr")
eda["corr_lf_late"] = float(corr.loc["Load factor", "Late rate"])
eda["corr_lf_distance"] = float(corr.loc["Load factor", "Distance"])
daily_late = op.groupby("flight_date")["is_late"].mean() * 100          # daily late-arrival rate
eda["corr_late_cancel"] = float(daily_late.corr(daily["rate"], method="spearman"))   # daily late rate vs daily cancellation rate
corr.round(2)

,Load factor,Late rate,Cancellation rate,Log flights,Distance,Departures
Load factor,1.00,0.10,-0.04,0.00,0.36,-0.00
Late rate,0.10,1.00,0.28,0.05,0.05,0.06
Cancellation rate,-0.04,0.28,1.00,0.21,-0.11,0.21
Log flights,0.00,0.05,0.21,1.00,-0.17,1.00
Distance,0.36,0.05,-0.11,-0.17,1.00,-0.17
Departures,-0.00,0.06,0.21,1.00,-0.17,1.00


## Figure 12 - Monthly passenger demand (RQ4 preview)

In [12]:
tot = series.sum(axis=1) / 1e6
fig, ax = plt.subplots(figsize=(6.5, 3.0))
years = sorted(tot.index.year.unique())
for i, y in enumerate(years):
    t = tot[tot.index.year == y]
    ax.plot(t.index.month, t.values, marker="o", ms=3.5, color=PALETTE[i % len(PALETTE)], label=str(y))
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTHS, rotation=45, ha="right"); ax.set_xlim(0.7, 12.3)
ax.set_xlabel("Month"); ax.set_ylabel("Passengers on the routes studied (millions)"); ax.legend(ncol=len(years), loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(); save_fig(fig, "fig12_demand_seasonality")
yearly = tot.groupby(tot.index.year).sum()
eda["demand_yearly_millions"] = {str(k): round(float(v), 2) for k, v in yearly.items()}
eda["demand_peak_month"] = int(tot.groupby(tot.index.month).mean().idxmax()); eda["demand_low_month"] = int(tot.groupby(tot.index.month).mean().idxmin())
m_avg = tot.groupby(tot.index.month).mean(); eda["demand_seasonal_swing_pct"] = float((m_avg.max() / m_avg.min() - 1) * 100)
save_result("eda", eda)
print("EDA complete; figures in", FIG)

EDA complete; figures in /Users/prathameshraut/Downloads/airline-delay-capstone/outputs/figures
